# Testing RoPE Implementation in hf transformers
* LlamaRotaryEmbedding [[link]](https://github.com/huggingface/transformers/blob/a5923d4de7df2fbd1f373dfcfe983216b79b6937/src/transformers/models/llama/modeling_llama.py#L71)
* modeling_rope_utils (_compute_default_rope_parameters) [[link]](https://github.com/huggingface/transformers/blob/2589a52c5cf60de8b96016f3267c98f0a1faa100/src/transformers/modeling_rope_utils.py#L92)


'''
position ids: (seq_len)
[0,1,...,seq_len-1]

inv_freq (base 10000): (dim//2)
[1.0000, 0.1000, 0.0100, 0.0010, ...]

-> freq (seq_len, dim//2)
einsum("i,j->ij", position_ids, inv_freq)

row: token position, column: dim_idx//2
[
    [po0/(base^dim1,2), pos0/(base^dim2,3), ...],
    [po1/(base^dim1,2), pos1/(base^dim2,3), ...],
    [po2/(base^dim1,2), pos2/(base^dim2,3), ...],
]

-> [1, seq_len, dim//2]

input x: (batch, seq_len, dim)
-> (batch, seq_len, dim//2) * 2

x_half * rotation
-> (batch, seq_len, dim//2) * (1, seq_len, 1, dim//2)
'''

For multi-head attention
```
freq matrix (seq_len, dim//2) -> (1, seq_len, 1, dim//2)

input: (batch, seq_len, **num_heads**, dim//2)
```

In [ ]:
import
from typing import Optional

# Init inv_freq
```
def _compute_default_rope_parameters(
    config: Optional[PretrainedConfig] = None,
    device: Optional["torch.device"] = None,
    seq_len: Optional[int] = None,
) -> tuple["torch.Tensor", float]:
    """
    Computes the inverse frequencies according to the original RoPE implementation
    Args:
        config ([`~transformers.PretrainedConfig`]):
            The model configuration.
        device (`torch.device`):
            The device to use for initialization of the inverse frequencies.
        seq_len (`int`, *optional*):
            The current sequence length. Unused for this type of RoPE.
    Returns:
        Tuple of (`torch.Tensor`, `float`), containing the inverse frequencies for the RoPE embeddings and the
        post-processing scaling factor applied to the computed cos/sin (unused in this type of RoPE).
    """
    base = config.rope_theta
    partial_rotary_factor = config.partial_rotary_factor if hasattr(config, "partial_rotary_factor") else 1.0
    head_dim = getattr(config, "head_dim", None) or config.hidden_size // config.num_attention_heads
    dim = int(head_dim * partial_rotary_factor)

    attention_factor = 1.0  # Unused in this type of RoPE

    # Compute the inverse frequencies
    inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.int64).to(device=device, dtype=torch.float) / dim))
    return inv_freq, attention_factor
```

In [20]:
base = 10000.0
# dim = 64
dim = 8

In [21]:
inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.int64).to(dtype=torch.float) / dim))

In [22]:
inv_freq

tensor([1.0000, 0.1000, 0.0100, 0.0010])

In [23]:
seq_len = 5
inv_freq_expanded = inv_freq[None, :, None].float().expand(seq_len, -1, 1)

In [28]:
inv_freq_expanded

tensor([[[1.0000],
         [0.1000],
         [0.0100],
         [0.0010]],

        [[1.0000],
         [0.1000],
         [0.0100],
         [0.0010]],

        [[1.0000],
         [0.1000],
         [0.0100],
         [0.0010]],

        [[1.0000],
         [0.1000],
         [0.0100],
         [0.0010]],

        [[1.0000],
         [0.1000],
         [0.0100],
         [0.0010]]])

In [24]:
inv_freq_expanded.shape # max_seq_len, dim//2, 1

torch.Size([5, 4, 1])

# RotaryEmbedding
rotation calculation

```
rotation matrix:s
[
    [cos, -sin],
    [sin, cos]
]

embedding pair [even, odd]
-> [cos*even - sin*odd, sin*even + cos*odd]

* new_even: cos*even - sin*odd
* new_odd: sin*even + cos*odd
```

In [25]:
# angle = token positon / (constant^(2k/d))

In [37]:
position_ids = torch.arange(seq_len)
position_ids

tensor([0, 1, 2, 3, 4])

In [39]:
freqs = torch.einsum("i,j->ij", position_ids, inv_freq)
freqs.shape # seq_len, dim//2

torch.Size([5, 4])

In [41]:
print(freqs)

tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 1.0000e-01, 1.0000e-02, 1.0000e-03],
        [2.0000e+00, 2.0000e-01, 2.0000e-02, 2.0000e-03],
        [3.0000e+00, 3.0000e-01, 3.0000e-02, 3.0000e-03],
        [4.0000e+00, 4.0000e-01, 4.0000e-02, 4.0000e-03]])


In [49]:
cos=freqs.cos()[None, :, :]
sin=freqs.sin()[None, :,  :]
print(cos.shape, sin.shape)

torch.Size([1, 5, 4]) torch.Size([1, 5, 4])


# Calculation

In [50]:
batch_size = 1
x = torch.randn(batch_size, seq_len, dim)

In [58]:
x_even = x[..., 0::2]
print(x_even.shape)

x_even = x_even.unsqueeze(-2)
print(x_even.shape)

torch.Size([1, 5, 4])
torch.Size([1, 5, 1, 4])


# 

In [51]:
x_even = x[:, :, 0::2]
x_odd = x[:, :, 1::2]

print(x_even.shape, x_odd.shape)

torch.Size([1, 5, 4]) torch.Size([1, 5, 4])


In [52]:
x_rot_even = x_even * cos - x_odd * sin
x_rot_odd  = x_even * sin + x_odd * cos

In [53]:
x_rot_even.shape, x_rot_odd.shape

(torch.Size([1, 5, 4]), torch.Size([1, 5, 4]))

In [54]:
x_out = torch.stack([x_rot_even, x_rot_odd], dim=-1)
print(x_out.shape)
# [batch, seq_len, num_heads, head_dim/2, 2]
x_out = x_out.flatten(-2)  # → [batch, seq_len, num_heads, head_dim]
print(x_out.shape)

torch.Size([1, 5, 4, 2])
torch.Size([1, 5, 8])


# Module

In [93]:
import torch
import torch.nn as nn


class RoPE(nn.Module):
    def __init__(
        self,
        theta: float,
        d_k: int,
        max_seq_len: int,
        device=None
    ):
        super().__init__()
        
        # initialize inv_freq
        dim_half_range = torch.arange(0, d_k, 2, dtype=torch.int64).to(dtype=torch.float)
        dim_half_range = dim_half_range/d_k
        
        inv_freq = 1.0 / (theta**dim_half_range)
        
        # register buffer
        self.register_buffer(
            'inv_freq',
            inv_freq,
            persistent=False # setting False excludes this from state_dict
        )
        
        
    def forward(
        self,
        x: torch.Tensor,
        token_positions: torch.Tensor
    ) -> torch.Tensor:
        '''
        x: shape (..., seq_len, d_k)
        token_positions: shape (..., seq_len)
        '''
        # Calculate Frequency
        # (batch, seq_len, d_k//2)
        # frq = token_positions@self.inv_freq
        # freq = torch.einsum("i,j->ij", token_positions, self.inv_freq)
        print('pos:', token_positions.shape)
        print('invfreq:', self.inv_freq.shape)
        freq = torch.einsum("... i, ... j->... ij", token_positions, self.inv_freq)
        print("freq:", freq.shape)
        
        # rotation
        cos = freq.cos()
        sin = freq.sin()
        
        # Split x (..., seq_len, d_k//2)
        x_even = x[..., 0::2]
        x_odd = x[..., 1::2]
        print("split x:", x_even.shape)
        
        # Rotate 
        x_new_even = x_even*cos - x_odd*sin
        x_new_odd = x_even*sin + x_odd*cos
        print("new x half:", x_new_even.shape)
        
        # Unsqueeze
        # on -1 to make (..., seq_len, d_k//2, 2)
        # so concat looks like [[even,odd], [even,odd], ...]
        x_new_even = x_new_even.unsqueeze(-1)
        x_new_odd = x_new_odd.unsqueeze(-1)
        print("unsqueezed new x half:", x_new_even.shape)
        
        # stack into (..., seq_len, d_k//2, 2) -> flatten
        x_out = torch.cat([x_new_even, x_new_odd], dim=-1)
        print("cat new x half:", x_out.shape)
        return x_out.flatten(-2)

In [94]:
theta = 10000.0
d_k = 8
max_seq_len = 5

rope = RoPE(
    theta=theta,
    d_k=d_k,
    max_seq_len=max_seq_len
)

In [95]:
seq_len = 3
batch_size = 2
x = torch.randn(batch_size, seq_len, dim)
token_positions = torch.arange(seq_len).repeat(batch_size, 1)

print(x.shape, token_positions.shape)

torch.Size([2, 3, 8]) torch.Size([2, 3])


In [96]:
token_positions

tensor([[0, 1, 2],
        [0, 1, 2]])

In [97]:
out = rope(x, token_positions)

pos: torch.Size([2, 3])
invfreq: torch.Size([4])
freq: torch.Size([2, 3, 4])
split x: torch.Size([2, 3, 4])
new x half: torch.Size([2, 3, 4])
unsqueezed new x half: torch.Size([2, 3, 4, 1])
cat new x half: torch.Size([2, 3, 4, 2])


In [98]:
print(out.shape)

torch.Size([2, 3, 8])
